# BTL02 T0204 — Activation ablation trên LargeCNN (Colab GPU)

Chọn runtime GPU trước khi chạy. Thêm `GITHUB_TOKEN` (repo private) và `KAGGLE_API_TOKEN` vào Colab Secrets; không dán token vào notebook. Chạy đủ 6 cell theo thứ tự. Mỗi cấu hình chỉ đổi activation; checkpoints, MLflow, summary CSV và epoch histories được lưu trong Google Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL02/T0204')
MLRUNS_DIR = DRIVE_ROOT / 'mlruns'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
LOGS_DIR = DRIVE_ROOT / 'logs'
for folder in (MLRUNS_DIR, CHECKPOINTS_DIR, LOGS_DIR): folder.mkdir(parents=True, exist_ok=True)
print('Artifacts will be saved under:', DRIVE_ROOT)


In [ ]:
import os, subprocess, tempfile
from pathlib import Path
from google.colab import userdata
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
BRANCH = 'exp/BTL02_T0204_ablation_activations'
REPO_DIR = Path('/content/Deep_Learning_Group')
token = userdata.get('GITHUB_TOKEN')
git_env = os.environ.copy()
askpass_path = None
if token:
    askpass = '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$COLAB_GITHUB_TOKEN" ;; esac\n'
    fd, askpass_path = tempfile.mkstemp(prefix='colab-git-askpass-')
    with os.fdopen(fd, 'w', encoding='utf-8') as stream: stream.write(askpass)
    os.chmod(askpass_path, 0o700)
    git_env.update({'GIT_ASKPASS': askpass_path, 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': token})
try:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True, env=git_env, capture_output=True, text=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True, env=git_env, capture_output=True, text=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True, env=git_env, capture_output=True, text=True)
finally:
    if askpass_path and os.path.exists(askpass_path): os.remove(askpass_path)
    git_env.pop('COLAB_GITHUB_TOKEN', None)
    token = None
PROJECT_DIR = REPO_DIR / 'BTL02'
os.chdir(PROJECT_DIR)
print('Checked out branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())


In [ ]:
import subprocess, sys
requirements = open('requirements.txt', encoding='utf-8').read().splitlines()
filtered = [line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=', 1)[0].strip().lower() not in {'torch', 'torchvision'}]
if sys.version_info >= (3, 13):
    filtered = ['scikit-learn==1.6.1' if line.strip().lower().startswith('scikit-learn==') else line for line in filtered]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *filtered], check=True)
import torch, torchvision
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật GPU runtime trong Colab trước khi huấn luyện.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os, subprocess, sys
from google.colab import userdata
kaggle_token = userdata.get('KAGGLE_API_TOKEN')
if not kaggle_token: raise RuntimeError('Thiếu Colab Secret KAGGLE_API_TOKEN.')
os.environ['KAGGLE_API_TOKEN'] = kaggle_token
del kaggle_token
subprocess.run([sys.executable, '-m', 'src.data.downloader'], check=True, env=os.environ.copy())
from src.data.downloader import FER2013Downloader
split_counts = FER2013Downloader.validate_csv(PROJECT_DIR / 'data/raw/fer2013.csv')
assert split_counts == {'Training': 28709, 'PublicTest': 3589, 'PrivateTest': 3589}, split_counts
print('FER2013 splits validated:', split_counts)


In [ ]:
import os, subprocess, sys
os.environ['MLFLOW_TRACKING_URI'] = f'file://{MLRUNS_DIR}'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
outputs_dir = PROJECT_DIR / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
def link_drive_folder(name, target):
    link = outputs_dir / name
    if link.is_symlink(): link.unlink()
    elif link.exists():
        items = list(link.iterdir()) if link.is_dir() else []
        if link.is_dir() and all(x.name == '.gitkeep' and x.is_file() for x in items):
            for item in items: item.unlink()
            link.rmdir()
        else: raise RuntimeError(f'Không ghi đè outputs/{name} có dữ liệu: {link}')
    link.symlink_to(target, target_is_directory=True)
    return link
link_drive_folder('checkpoints', CHECKPOINTS_DIR)
link_drive_folder('logs', LOGS_DIR)
print('MLflow URI:', os.environ['MLFLOW_TRACKING_URI'])
configs = [
 'experiments/ablation/exp07_act_relu.yaml',
 'experiments/ablation/exp07_act_leaky_relu.yaml',
 'experiments/ablation/exp07_act_gelu.yaml',
 'experiments/ablation/exp07_act_elu.yaml',
]
for config in configs:
    print(f'\n===== {config} =====', flush=True)
    subprocess.run([sys.executable, '-u', '-m', 'src.train.trainer', '--config', config], check=True, env=os.environ.copy())


In [ ]:
import os, pandas as pd, mlflow
from mlflow.tracking import MlflowClient
from src.train.mlflow_logger import MLflowLogger
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
logger = MLflowLogger()
csv_path = DRIVE_ROOT / 't0204_ablation_activations_24col.csv'
raw_path = LOGS_DIR / 't0204_all_mlflow_runs_24col.csv'
logger.export_summary_csv(str(raw_path))
summary = pd.read_csv(raw_path, encoding='utf-8-sig')
assert len(summary.columns) == 24, f'Expected 24 columns, got {len(summary.columns)}'
expected = {'relu', 'leaky_relu', 'gelu', 'elu'}
summary['_start_sort'] = pd.to_datetime(summary['Start Time'], errors='coerce', utc=True)
summary = summary[(summary['params.exp_group'].astype(str) == 'EXP-07') & (summary['params.rq_tag'].astype(str) == 'RQ1_Ablation_Activation') & (summary['params.architecture'].astype(str) == 'LargeCNN')]
latest_rows = []
for activation in sorted(expected):
    candidates = summary[summary['Name'].astype(str).str.endswith('_' + activation)].sort_values('_start_sort')
    if candidates.empty: raise RuntimeError(f'Không tìm thấy MLflow run cho activation={activation}')
    latest = candidates.iloc[-1]
    if str(latest['Status']).upper() != 'FINISHED': raise RuntimeError(f'Run mới nhất của {activation} không FINISHED: {latest["Status"]}')
    latest_rows.append(latest.drop(labels=['_start_sort']))
finished = pd.DataFrame(latest_rows).reindex(columns=MLflowLogger.CSV_COLUMNS)
assert len(finished) == 4 and len(finished.columns) == 24
finished.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('24-column summary saved:', csv_path)
display(finished[['Name', 'Status', 'Run ID', 'metrics.best_epoch', 'metrics.train_loss', 'metrics.train_acc', 'metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1']])
client = MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI'])
history_rows = []
for _, run in finished.iterrows():
    rid = run['Run ID']
    for metric_name in ('train_loss', 'val_loss', 'train_acc', 'val_acc'):
        for item in client.get_metric_history(rid, metric_name):
            if item.step > 0: history_rows.append({'run_id': rid, 'run_name': run['Name'], 'metric': metric_name, 'epoch': item.step, 'value': item.value})
history_path = DRIVE_ROOT / 't0204_epoch_metrics.csv'
pd.DataFrame(history_rows).to_csv(history_path, index=False)
print('Epoch curves saved:', history_path, '| rows:', len(history_rows))
print('Checkpoints:', CHECKPOINTS_DIR, '| MLflow:', MLRUNS_DIR)
